# Session 09 Solutions: Reading Files and Handling Errors

Data Science with Python, Level 1, Stage 1: Python Foundations

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## In-Class Exercises

### In-Class Exercise 1: Diary

In [1]:
with open("diary.txt", "w") as f:
    f.write("Woke up early\n")
    f.write("Studied Python\n")
    f.write("Went for a walk\n")
with open("diary.txt", "a") as f:
    f.write("Read about files\n")
with open("diary.txt") as f:
    for n, line in enumerate(f, start=1):
        print(n, line.strip())

1 Woke up early
2 Studied Python
3 Went for a walk
4 Read about files


### In-Class Exercise 2: Safe Input

In [2]:
def to_number(text):
    try:
        return float(text)
    except ValueError:
        return None

values = ["3", "4.5", "abc", "", "10"]
numbers = [to_number(v) for v in values if to_number(v) is not None]
print(numbers, sum(numbers))

[3.0, 4.5, 10.0] 17.5


### In-Class Exercise 3: Average from a File

In [3]:
with open("scores.txt", "w") as f:
    for s in [72, 88, 95, 64, 81]:
        f.write(f"{s}\n")
with open("scores.txt") as f:
    scores = [int(line) for line in f]
print(len(scores), sum(scores) / len(scores))

5 80.0


### In-Class Exercise 4: Validate an Age

In [4]:
def check_age(age):
    if age < 0 or age > 120:
        raise ValueError(f"impossible age: {age}")
    return age

for a in [25, -3, 230]:
    try:
        print("OK:", check_age(a))
    except ValueError as e:
        print("Error:", e)

OK: 25
Error: impossible age: -3
Error: impossible age: 230


## Part A: Basic

### Exercise 1: Shopping List File

In [5]:
items = ["milk", "bread", "eggs", "coffee", "apples"]
with open("shopping.txt", "w") as f:
    for item in items:
        f.write(item + "\n")
with open("shopping.txt") as f:
    loaded = [line.strip() for line in f]
for n, item in enumerate(sorted(loaded), start=1):
    print(n, item)

1 apples
2 bread
3 coffee
4 eggs
5 milk


### Exercise 2: Name the Error

In [6]:
try:
    int("3.5")
except ValueError as e:
    print(type(e).__name__, e)
try:
    [10, 20, 30][3]
except IndexError as e:
    print(type(e).__name__, e)
try:
    {"a": 1}["b"]
except KeyError as e:
    print(type(e).__name__, e)
try:
    "total: " + 5
except TypeError as e:
    print(type(e).__name__, e)

ValueError invalid literal for int() with base 10: '3.5'
IndexError list index out of range
KeyError 'b'
TypeError can only concatenate str (not "int") to str


## Part B: Intermediate

### Exercise 3: Temperature Log

`setdefault(city, [])` returns the list for a city, creating an empty one the first time the city appears.

In [7]:
content = "city,temp\nDamascus,24.5\nAleppo,22.1\nDamascus,26.0\nAleppo,n/a\nDamascus,25.2\nAleppo,23.4\n"
with open("temps.csv", "w") as f:
    f.write(content)

temps = {}
with open("temps.csv") as f:
    f.readline()
    for line in f:
        city, value = line.strip().split(",")
        try:
            t = float(value)
        except ValueError:
            print(f"Warning: skipped {line.strip()!r}")
            continue
        temps.setdefault(city, []).append(t)
for city, values in temps.items():
    print(f"{city}: {sum(values) / len(values):.1f}")

Damascus: 25.2
Aleppo: 22.8


### Exercise 4: Word Count of a File

In [8]:
with open("paragraph.txt", "w") as f:
    f.write("Data science starts with questions.\n")
    f.write("Good data helps us answer them.\n")
    f.write("Python makes the work faster.\n")

def file_stats(filename):
    try:
        with open(filename) as f:
            text = f.read()
    except FileNotFoundError:
        print(f"{filename} does not exist")
        return None
    return {"lines": len(text.splitlines()), "words": len(text.split()), "characters": len(text)}

print(file_stats("paragraph.txt"))
print(file_stats("nothing.txt"))

{'lines': 3, 'words': 16, 'characters': 98}
nothing.txt does not exist
None


## Part C: Challenge

### Exercise 5: Clean a Messy CSV

In [9]:
content = "name,age,score\nLina,23,88\nOmar,,75\nSara,twenty,91\nHadi,31,\nMaya,27,64\n"
with open("survey.csv", "w") as f:
    f.write(content)

clean, rejected = [], []
with open("survey.csv") as f:
    f.readline()
    for line in f:
        name, age, score = line.strip().split(",")
        try:
            clean.append({"name": name, "age": int(age), "score": int(score)})
        except ValueError as e:
            rejected.append((line.strip(), str(e)))

print(clean)
for line, reason in rejected:
    print("Rejected:", line, "|", reason)
print("Average score:", sum(r["score"] for r in clean) / len(clean))

[{'name': 'Lina', 'age': 23, 'score': 88}, {'name': 'Maya', 'age': 27, 'score': 64}]
Rejected: Omar,,75 | invalid literal for int() with base 10: ''
Rejected: Sara,twenty,91 | invalid literal for int() with base 10: 'twenty'
Rejected: Hadi,31, | invalid literal for int() with base 10: ''
Average score: 76.0


## Part D: Cumulative Project: Campus Bookshop

### Exercise 6: Save and Load the Bookshop Catalogue

In [10]:
catalogue = {
    "Python Basics": {"price": 45.0, "stock": 12, "category": "Programming"},
    "Data Science Handbook": {"price": 62.5, "stock": 3, "category": "Data"},
    "SQL in a Week": {"price": 29.9, "stock": 7, "category": "Programming"},
}

def save_catalogue(catalogue, filename):
    with open(filename, "w") as f:
        f.write("title,price,stock,category\n")
        for title, b in catalogue.items():
            f.write(f"{title},{b['price']},{b['stock']},{b['category']}\n")

def load_catalogue(filename):
    result = {}
    try:
        with open(filename) as f:
            f.readline()
            for line in f:
                title, price, stock, category = line.strip().split(",")
                try:
                    result[title] = {"price": float(price), "stock": int(stock), "category": category}
                except ValueError:
                    print("Skipped:", line.strip())
    except FileNotFoundError:
        print(f"{filename} not found")
    return result

save_catalogue(catalogue, "catalogue.csv")
with open("catalogue.csv", "a") as f:
    f.write("Broken Book,free,3,Data\n")
loaded = load_catalogue("catalogue.csv")
for title, b in loaded.items():
    print(title, b)
print(load_catalogue("no_such_file.csv"))

Skipped: Broken Book,free,3,Data
Python Basics {'price': 45.0, 'stock': 12, 'category': 'Programming'}
Data Science Handbook {'price': 62.5, 'stock': 3, 'category': 'Data'}
SQL in a Week {'price': 29.9, 'stock': 7, 'category': 'Programming'}
no_such_file.csv not found
{}


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | C | `"a"` appends; `"w"` would replace the file. |
| 2 | C | Text files always give strings; you convert them yourself. |
| 3 | B | The type is right (a string) but the value cannot be converted. |
| 4 | C | `finally` runs whether or not an error happened. |
| 5 | B | A bare `except` hides every error, including bugs you did not expect. |